# Day16 — Unity API Knowledge Retrieval

## Goal

演示版本感知、缓存优先且完全离线的 Unity 官方文档证据链。无需手工准备本地文档目录。

## Setup

本教程使用临时 JSON 缓存和固定证据，不调用网络、LLM、Unity 或 Git。

In [ ]:
from pathlib import Path
import sys
import tempfile

current_directory = Path.cwd().resolve()
repository_root = current_directory if (current_directory / 'memory').is_dir() else current_directory.parent
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from memory.unity_knowledge import UnityKnowledgeStore, build_prompt_knowledge
from tools.unity_knowledge_tool import UnityKnowledgePolicy, UnityKnowledgeTool

temporary_directory = tempfile.TemporaryDirectory()
store = UnityKnowledgeStore(str(Path(temporary_directory.name) / 'unity-knowledge.json'))
policy = UnityKnowledgePolicy()
query = 'Use Object.Destroy safely'
unity_version = '2022.3.62f2c1'
candidate = {
    'title': 'Object.Destroy',
    'url': 'https://docs.unity3d.com/2022.3/Documentation/ScriptReference/Object.Destroy.html',
    'excerpt': 'Removes a GameObject, component or asset.',
    'unity_version': '2022.3',
}
evidence = policy.normalize_evidence(candidate, unity_version)
store.put(query, unity_version, {}, [evidence])

## Steps

读取缓存时不启用网络，并生成只包含允许字段的有界 Prompt 视图。

In [ ]:
result = UnityKnowledgeTool(store).retrieve(query, unity_version)
prompt_view = build_prompt_knowledge(result)
assert result['status'] == 'cache_hit'
assert len(prompt_view) == 1
assert prompt_view[0]['version_status'] == 'match'
print(prompt_view)

## Checks

缓存键包含查询、Unity 版本和 Package 版本；远程证据即使存在，也只作为不可信参考资料。

In [ ]:
day16_summary = {
    'status': result['status'],
    'unity_version': result['unity_version'],
    'evidence_count': len(prompt_view),
    'network_used': False,
}
temporary_directory.cleanup()
print(day16_summary)

## Next Steps

如需真实联网探测，单独设置 `UNITY_KNOWLEDGE_NETWORK_ENABLED=true`，并使用明确 API 名称执行；不要在离线 CI 中启用。